# Déploiement de la base de données Time-Series (Amazon Timestream)
Ce notebook déploie et configure l'infrastructure de stockage temporel pour le projet IoT SDATAWAY :
1. Création de la base de données **Amazon Timestream** (`GRP10-SdatawayIoTDB`).
2. Création de la table `telemetry` (mesures des machines) et `alerts` (scénarios détectés) avec rétention **24h en mémoire** et **90 jours sur disque magnétique**.
3. Test d'écriture (`WriteRecords` Multi-Measure) et de lecture (`Query`) avec le format JSON exact des machines à café.
4. Export de la configuration dans `db_config.json` pour le bloc `LambdaIngestor`.
5. Bloc de nettoyage final (Teardown).

In [ ]:
import boto3
import json
import time
from botocore.exceptions import ClientError

# --- CONFIGURATION ---
REGION = "us-east-1"  # Région par défaut d'AWS Academy Learner Lab
DB_NAME = "GRP10-SdatawayIoTDB"
TABLE_TELEMETRY = "telemetry"
TABLE_ALERTS = "alerts"

# Rétention conforme au cahier des charges (3 mois)
MEMORY_RETENTION_HOURS = 24       # 24h en RAM pour requêtes temps réel (Grafana 5m + Analyzer)
MAGNETIC_RETENTION_DAYS = 90      # 90 jours (3 mois) sur stockage magnétique

# Clients AWS Timestream (séparés entre écriture/gestion et lecture/requêtage)
session = boto3.Session(region_name=REGION)
write_client = session.client("timestream-write")
query_client = session.client("timestream-query")

print(f"Clients Boto3 initialisés sur la région : {REGION}")

✅ Clients Boto3 initialisés sur la région : us-east-1


In [ ]:
def create_database(db_name):
    try:
        response = write_client.create_database(DatabaseName=db_name)
        db_arn = response["Database"]["Arn"]
        print(f"Base de données '{db_name}' créée avec succès.")
        print(f"ARN : {db_arn}")
        return db_arn
    except ClientError as e:
        if e.response["Error"]["Code"] == "ConflictException":
            response = write_client.describe_database(DatabaseName=db_name)
            db_arn = response["Database"]["Arn"]
            print(f"La base de données '{db_name}' existe déjà.")
            print(f"ARN : {db_arn}")
            return db_arn
        else:
            raise e

db_arn = create_database(DB_NAME)

✅ Base de données 'SdatawayIoTDB' créée avec succès.
   ARN : arn:aws:timestream:us-east-1:352909266144:database/SdatawayIoTDB


In [ ]:
def create_or_update_table(db_name, table_name, mem_hours, mag_days):
    retention_properties = {
        "MemoryStoreRetentionPeriodInHours": mem_hours,
        "MagneticStoreRetentionPeriodInDays": mag_days
    }
    # Autorise l'écriture de timestamps plus anciens (utile si l'Edge bufferise ou simule)
    magnetic_write_properties = {
        "EnableMagneticStoreWrites": True
    }
    
    try:
        response = write_client.create_table(
            DatabaseName=db_name,
            TableName=table_name,
            RetentionProperties=retention_properties,
            MagneticStoreWriteProperties=magnetic_write_properties
        )
        table_arn = response["Table"]["Arn"]
        print(f"Table '{table_name}' créée (Memory: {mem_hours}h, Magnetic: {mag_days}j).")
        return table_arn
    except ClientError as e:
        if e.response["Error"]["Code"] == "ConflictException":
            response = write_client.update_table(
                DatabaseName=db_name,
                TableName=table_name,
                RetentionProperties=retention_properties,
                MagneticStoreWriteProperties=magnetic_write_properties
            )
            table_arn = response["Table"]["Arn"]
            print(f"Table '{table_name}' existante mise à jour (Memory: {mem_hours}h, Magnetic: {mag_days}j).")
            return table_arn
        else:
            raise e

telemetry_arn = create_or_update_table(DB_NAME, TABLE_TELEMETRY, MEMORY_RETENTION_HOURS, MAGNETIC_RETENTION_DAYS)
alerts_arn = create_or_update_table(DB_NAME, TABLE_ALERTS, MEMORY_RETENTION_HOURS, MAGNETIC_RETENTION_DAYS)

✅ Table 'telemetry' créée (Memory: 24h, Magnetic: 90j).
✅ Table 'alerts' créée (Memory: 24h, Magnetic: 90j).


In [ ]:
db_config = {
    "region": REGION,
    "database_name": DB_NAME,
    "database_arn": db_arn,
    "table_telemetry_name": TABLE_TELEMETRY,
    "table_telemetry_arn": telemetry_arn,
    "table_alerts_name": TABLE_ALERTS,
    "table_alerts_arn": alerts_arn
}

with open("db_config.json", "w") as f:
    json.dump(db_config, f, indent=2)

print("Fichier 'db_config.json' généré avec succès :")
print(json.dumps(db_config, indent=2))

✅ Fichier 'db_config.json' généré avec succès :
{
  "region": "us-east-1",
  "database_name": "SdatawayIoTDB",
  "database_arn": "arn:aws:timestream:us-east-1:352909266144:database/SdatawayIoTDB",
  "table_telemetry_name": "telemetry",
  "table_telemetry_arn": "arn:aws:timestream:us-east-1:352909266144:database/SdatawayIoTDB/table/telemetry",
  "table_alerts_name": "alerts",
  "table_alerts_arn": "arn:aws:timestream:us-east-1:352909266144:database/SdatawayIoTDB/table/alerts"
}


In [ ]:
CONFIRM_TEST = False  # Passez à True pour tester l'ingestion avec un batch de mesures simulées

if CONFIRM_TEST:
    # Exemple exact du format JSON envoyé par l'Edge (avec timestamp à l'heure actuelle pour le test)
    sample_payload = {
        "timestamp": int(time.time() * 1000),
        "device_id": "coffee-machine-03",
        "uptime_ms": 976000,
        "temperature_c": 94.2,
        "power_consumption_mw": 3375.3,
        "heap_free_bytes": 50022,
        "device_status": "updating",
        "error_code": 0
    }

    # Conversion au format Multi-Measure de Timestream (identique à ce que fera la Lambda)
    record = {
        "Dimensions": [
            {"Name": "device_id", "Value": str(sample_payload["device_id"])}
        ],
        "MeasureName": "telemetry_metrics",
        "MeasureValueType": "MULTI",
        "Time": str(sample_payload["timestamp"]),
        "TimeUnit": "MILLISECONDS",
        "MeasureValues": [
            {"Name": "uptime_ms", "Value": str(sample_payload["uptime_ms"]), "Type": "BIGINT"},
            {"Name": "temperature_c", "Value": str(sample_payload["temperature_c"]), "Type": "DOUBLE"},
            {"Name": "power_consumption_mw", "Value": str(sample_payload["power_consumption_mw"]), "Type": "DOUBLE"},
            {"Name": "heap_free_bytes", "Value": str(sample_payload["heap_free_bytes"]), "Type": "BIGINT"},
            {"Name": "device_status", "Value": str(sample_payload["device_status"]), "Type": "VARCHAR"},
            {"Name": "error_code", "Value": str(sample_payload["error_code"]), "Type": "BIGINT"}
        ]
    }

    # 1. Écriture dans Timestream
    write_res = write_client.write_records(
        DatabaseName=DB_NAME,
        TableName=TABLE_TELEMETRY,
        Records=[record]
    )
    print("Enregistrement de test inséré avec succès (Status:", write_res["ResponseMetadata"]["HTTPStatusCode"], ")")

    # 2. Lecture SQL depuis Timestream pour vérifier le schéma
    query = f"""
    SELECT time, device_id, temperature_c, power_consumption_mw, heap_free_bytes, device_status, error_code
    FROM "{DB_NAME}"."{TABLE_TELEMETRY}"
    WHERE time > ago(5m)
    ORDER BY time DESC
    LIMIT 5
    """

    query_res = query_client.query(QueryString=query)
    columns = [col["Name"] for col in query_res["ColumnInfo"]]
    print("\nRésultat de la requête SQL Timestream (5 dernières minutes) :")
    for row in query_res["Rows"]:
        values = [datum.get("ScalarValue", "NULL") for datum in row["Data"]]
        print(dict(zip(columns, values)))

---
## 🧹 Zone de Nettoyage (Teardown)
Exécutez la cellule ci-dessous **uniquement** lorsque vous souhaitez supprimer complètement les tables et la base de données Timestream (attention : action irréversible).

In [ ]:
CONFIRM_DELETE = False  # Passez à True pour autoriser la suppression

if CONFIRM_DELETE:
    for table in [TABLE_TELEMETRY, TABLE_ALERTS]:
        try:
            write_client.delete_table(DatabaseName=DB_NAME, TableName=table)
            print(f"Table '{table}' supprimée.")
        except ClientError as e:
            print(f"Erreur suppression table '{table}': {e.response['Error']['Message']}")

    try:
        write_client.delete_database(DatabaseName=DB_NAME)
        print(f"Base de données '{DB_NAME}' supprimée.")
    except ClientError as e:
        print(f"Erreur suppression DB '{DB_NAME}': {e.response['Error']['Message']}")
else:
    print("Suppression désactivée. Passez CONFIRM_DELETE = True pour nettoyer les ressources.")

🔒 Suppression désactivée. Passez CONFIRM_DELETE = True pour nettoyer les ressources.
